# DIT5411 Traditional Chinese Handwriting Recognition

This notebook supports both **Google Colab** and **local Windows execution**. It downloads and verifies the dataset, splits train/validation/test data, performs on-the-fly augmentation, trains three CNN models, compares test accuracy, and saves predictions.

## Google Colab

1. Select **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Select **Runtime → Run all** and allow Google Drive access.
3. Outputs are saved to `My Drive/DIT5411-Au-YeungManChingRachel/`.

## Local Windows

1. Keep the dataset at `data/all_data.zip`. The notebook downloads it automatically if missing.
2. Open this notebook in VS Code or Jupyter using the project's `.venv` Python environment.
3. Run all cells. No Google Drive connection is required.
4. Outputs are saved to `outputs/DIT5411_local/`.

The full run trains three models for up to 10 epochs each. Local execution uses CPU when no compatible GPU is available and can take a long time. Checkpoints are saved after every epoch. To resume, run all cells again without deleting the output directory; completed models are skipped and incomplete models resume from their latest checkpoint.

For a quick pipeline check only, change `SMOKE_TEST = False` to `True` in the next cell. Smoke-test accuracy is not a valid assignment result.

## Apple Silicon Mac

Use a native ARM64 Python environment (not Rosetta). Install Xcode command-line tools with `xcode-select --install`, then run:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip
python -m pip install -r requirements.txt
```

On Apple Silicon, the notebook detects `darwin` + `arm64`, ensures the `tensorflow-metal` plug-in is installed, and uses the Metal GPU when TensorFlow reports a GPU. Outputs are saved to `outputs/DIT5411_local/`. If the plug-in was installed while an old kernel was running, restart the kernel before running all cells.


In [ ]:
import csv
import gc
import json
import math
import platform
import re
import subprocess
import sys
import time
from pathlib import Path, PurePosixPath
from zipfile import BadZipFile, ZipFile

IS_APPLE_SILICON = sys.platform == "darwin" and platform.machine() == "arm64"
if IS_APPLE_SILICON:
    from importlib.metadata import PackageNotFoundError, version as package_version
    try:
        package_version("tensorflow-metal")
    except PackageNotFoundError:
        print("Installing TensorFlow Metal plug-in for Apple Silicon...")
        subprocess.run([
            sys.executable, "-m", "pip", "install", "tensorflow-metal"
        ], check=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

SMOKE_TEST = False
EXPECTED_CLASSES = 13_065
EXPECTED_IMAGES = 684_677
IMAGE_SIZE = 64
MIN_AUGMENTED_SAMPLES = 200
STANDARD_TRAIN_SAMPLES = 40
BATCH_SIZE = 16 if SMOKE_TEST else 256
EPOCHS = 1 if SMOKE_TEST else 10
MAX_CLASSES = 8 if SMOKE_TEST else None
STEPS_PER_EPOCH = 2 if SMOKE_TEST else None
VALIDATION_STEPS = 2 if SMOKE_TEST else None
EVALUATION_STEPS = 2 if SMOKE_TEST else None
VERIFY_CRC = not SMOKE_TEST
SEED = 42

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB and not SMOKE_TEST:
    BATCH_SIZE = 64

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content")
    ARCHIVE_PATH = WORK_DIR / "all_data.zip"
    REPOSITORY_DIR = WORK_DIR / "handwritting_data_all"
    OUTPUT_DIR = Path("/content/drive/MyDrive/DIT5411-Au-YeungManChingRachel")
else:
    WORK_DIR = Path.cwd()
    if not (WORK_DIR / "data" / "all_data.zip").exists() and (WORK_DIR.parent / "data" / "all_data.zip").exists():
        WORK_DIR = WORK_DIR.parent
    ARCHIVE_PATH = WORK_DIR / "data" / "all_data.zip"
    REPOSITORY_DIR = WORK_DIR / "data" / "handwritting_data_all"
    output_name = "colab_notebook_smoke" if SMOKE_TEST else "DIT5411_local"
    OUTPUT_DIR = WORK_DIR / "outputs" / output_name

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

np.random.seed(SEED)
tf.random.set_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")

if IN_COLAB:
    run_mode = "Colab"
elif IS_APPLE_SILICON:
    run_mode = "Local Apple Silicon"
else:
    run_mode = "Local"

print("Run mode:", run_mode)
print("Platform:", platform.platform())
print("Machine:", platform.machine())
print("Batch size:", BATCH_SIZE)
print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("GPUs:", gpus)
print("Smoke test:", SMOKE_TEST)
print("Output directory:", OUTPUT_DIR)
if IS_APPLE_SILICON:
    print("Apple Silicon Metal GPU:", bool(gpus))
    if not gpus:
        print("Metal GPU not detected. Reinstall tensorflow-metal and restart the kernel.")
if not gpus:
    print("WARNING: No GPU is visible. Full training will use CPU and may be slow.")

In [ ]:
ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)


def archive_is_valid(path):
    if not path.exists():
        return False
    try:
        with ZipFile(path) as archive:
            return len(archive.infolist()) > 0
    except (BadZipFile, OSError):
        return False

if archive_is_valid(ARCHIVE_PATH):
    print("Using existing archive:", ARCHIVE_PATH)
else:
    if not REPOSITORY_DIR.exists():
        print("Downloading dataset repository...")
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/chenkenanalytic/handwritting_data_all.git",
            str(REPOSITORY_DIR),
        ], check=True)

    parts = [REPOSITORY_DIR / f"all_data.zip.{i:03d}" for i in range(1, 13)]
    missing = [str(part) for part in parts if not part.exists()]
    assert not missing, f"Missing archive parts: {missing}"

    temporary_archive = ARCHIVE_PATH.with_suffix(".zip.tmp")
    with temporary_archive.open("wb") as destination:
        for part in parts:
            print("Merging", part.name)
            with part.open("rb") as source:
                while chunk := source.read(16 * 1024 * 1024):
                    destination.write(chunk)
    temporary_archive.replace(ARCHIVE_PATH)

assert archive_is_valid(ARCHIVE_PATH), "The merged dataset is not a valid ZIP archive"
print("Archive size: %.2f GB" % (ARCHIVE_PATH.stat().st_size / 1e9))

In [ ]:
with ZipFile(ARCHIVE_PATH) as archive:
    infos = archive.infolist()
    files = [info for info in infos if not info.is_dir()]
    archive_class_ids = {
        PurePosixPath(info.filename).parts[1]
        for info in files
    }
    print("Archive entries:", len(infos))
    print("Image files:", len(files))
    print("Character classes:", len(archive_class_ids))
    assert len(files) == EXPECTED_IMAGES
    assert len(archive_class_ids) == EXPECTED_CLASSES
    if VERIFY_CRC:
        print("Checking every ZIP entry (this can take a few minutes)...")
        bad_file = archive.testzip()
        assert bad_file is None, f"Corrupt archive entry: {bad_file!r}"

print("Dataset verification passed.")

In [ ]:
groups = {}
with ZipFile(ARCHIVE_PATH) as archive:
    for info in archive.infolist():
        if info.is_dir():
            continue
        parts = PurePosixPath(info.filename).parts
        assert len(parts) >= 3 and parts[0] == "cleaned_data" and parts[1].isdigit()
        match = re.search(r"_(\d+)\.png$", parts[-1], re.IGNORECASE)
        assert match is not None, info.filename
        groups.setdefault(parts[1], []).append((int(match.group(1)), info.filename))

all_class_ids = sorted(groups, key=int)
selected_class_ids = all_class_ids[:MAX_CLASSES] if MAX_CLASSES else all_class_ids
classes = []

for label, class_id in enumerate(selected_class_ids):
    ordered = sorted(groups[class_id], key=lambda item: (item[0], item[1]))
    if len(ordered) >= STANDARD_TRAIN_SAMPLES:
        train_count = STANDARD_TRAIN_SAMPLES
    else:
        train_count = max(2, round(len(ordered) * 0.8))

    train_and_validation = ordered[:train_count]
    test = ordered[train_count:]
    validation_count = max(1, round(len(train_and_validation) * 0.1))
    fit = train_and_validation[:-validation_count]
    validation = train_and_validation[-validation_count:]
    repeat_count = math.ceil(MIN_AUGMENTED_SAMPLES / len(fit))

    classes.append({
        "class_id": class_id,
        "label": label,
        "fit": fit,
        "validation": validation,
        "test": test,
        "repeat_count": repeat_count,
    })

num_classes = len(classes)
class_mapping = {str(item["label"]): item["class_id"] for item in classes}
with (OUTPUT_DIR / "class_mapping.json").open("w", encoding="utf-8") as file:
    json.dump(class_mapping, file, ensure_ascii=False, indent=2)

summary = {
    "classes": num_classes,
    "fit_images": sum(len(item["fit"]) for item in classes),
    "validation_images": sum(len(item["validation"]) for item in classes),
    "test_images": sum(len(item["test"]) for item in classes),
    "augmented_fit_samples_per_epoch": sum(
        len(item["fit"]) * item["repeat_count"] for item in classes
    ),
    "minimum_augmented_fit_samples_per_class": min(
        len(item["fit"]) * item["repeat_count"] for item in classes
    ),
    "classes_with_fewer_than_40_images": sum(
        len(groups[class_id]) < STANDARD_TRAIN_SAMPLES
        for class_id in selected_class_ids
    ),
}
with (OUTPUT_DIR / "dataset_summary.json").open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print(json.dumps(summary, indent=2))
assert summary["minimum_augmented_fit_samples_per_class"] >= MIN_AUGMENTED_SAMPLES
assert all(item["test"] for item in classes)

In [ ]:
fit_count = sum(len(item["fit"]) for item in classes)
validation_count = sum(len(item["validation"]) for item in classes)
test_count = sum(len(item["test"]) for item in classes)

fit_images = np.empty((fit_count, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
validation_images = np.empty((validation_count, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
test_images = np.empty((test_count, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
validation_labels = np.empty(validation_count, dtype=np.uint16)
test_labels = np.empty(test_count, dtype=np.uint16)

fit_cursor = validation_cursor = test_cursor = 0
loaded = 0
started = time.time()

def decode_image(archive, archive_name):
    data = np.frombuffer(archive.read(archive_name), dtype=np.uint8)
    image = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Cannot decode {archive_name!r}")
    return cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

with ZipFile(ARCHIVE_PATH) as archive:
    for item in classes:
        item["fit_start"] = fit_cursor
        for _, archive_name in item["fit"]:
            fit_images[fit_cursor] = decode_image(archive, archive_name)
            fit_cursor += 1
            loaded += 1
            if loaded % 50_000 == 0:
                print(f"Loaded {loaded:,} images in {(time.time() - started) / 60:.1f} minutes")

        for _, archive_name in item["validation"]:
            validation_images[validation_cursor] = decode_image(archive, archive_name)
            validation_labels[validation_cursor] = item["label"]
            validation_cursor += 1
            loaded += 1

        for _, archive_name in item["test"]:
            test_images[test_cursor] = decode_image(archive, archive_name)
            test_labels[test_cursor] = item["label"]
            test_cursor += 1
            loaded += 1

assert fit_cursor == fit_count
assert validation_cursor == validation_count
assert test_cursor == test_count

plan_size = sum(len(item["fit"]) * item["repeat_count"] for item in classes)
fit_positions = np.empty(plan_size, dtype=np.int32)
fit_labels = np.empty(plan_size, dtype=np.uint16)
plan_cursor = 0

for item in classes:
    start = item["fit_start"]
    count = len(item["fit"])
    positions = np.arange(start, start + count, dtype=np.int32)
    for _ in range(item["repeat_count"]):
        end = plan_cursor + count
        fit_positions[plan_cursor:end] = positions
        fit_labels[plan_cursor:end] = item["label"]
        plan_cursor = end

assert plan_cursor == plan_size
order = np.random.default_rng(SEED).permutation(plan_size)
fit_positions = fit_positions[order]
fit_labels = fit_labels[order]
del order

print(f"Loaded {loaded:,} images in {(time.time() - started) / 60:.1f} minutes")
print("Image arrays use %.2f GB" % (
    (fit_images.nbytes + validation_images.nbytes + test_images.nbytes) / 1e9
))
print("Augmented fit plan:", fit_positions.shape)

del groups
gc.collect()

In [ ]:
fit_image_tensor = tf.constant(fit_images, dtype=tf.uint8)
validation_image_tensor = tf.constant(validation_images, dtype=tf.uint8)
test_image_tensor = tf.constant(test_images, dtype=tf.uint8)

del fit_images, validation_images, test_images, classes
gc.collect()

fit_dataset = tf.data.Dataset.from_tensor_slices((fit_positions, fit_labels))
fit_dataset = fit_dataset.shuffle(
    min(len(fit_positions), 100_000), seed=SEED, reshuffle_each_iteration=True
)

def lookup_fit_image(position, label):
    image = tf.gather(fit_image_tensor, position)
    return image[..., tf.newaxis], tf.cast(label, tf.int32)

def add_channel(image, label):
    return image[..., tf.newaxis], tf.cast(label, tf.int32)

fit_dataset = fit_dataset.map(lookup_fit_image, num_parallel_calls=tf.data.AUTOTUNE)
fit_dataset = fit_dataset.batch(BATCH_SIZE)

validation_dataset = tf.data.Dataset.from_tensor_slices((validation_image_tensor, validation_labels))
validation_dataset = validation_dataset.map(add_channel, num_parallel_calls=tf.data.AUTOTUNE)
validation_dataset = validation_dataset.batch(BATCH_SIZE)

test_dataset = tf.data.Dataset.from_tensor_slices((test_image_tensor, test_labels))
test_dataset = test_dataset.map(add_channel, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE)

augmenter = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(
        0.03, fill_mode="constant", fill_value=1.0, seed=SEED
    ),
    tf.keras.layers.RandomZoom(
        height_factor=(-0.1, 0.1), width_factor=(-0.1, 0.1),
        fill_mode="constant", fill_value=1.0, seed=SEED + 1
    ),
    tf.keras.layers.RandomShear(
        x_factor=(0.0, 0.12), fill_mode="constant", fill_value=1.0, seed=SEED + 2
    ),
    tf.keras.layers.RandomTranslation(
        height_factor=0.03, width_factor=0.03,
        fill_mode="constant", fill_value=1.0, seed=SEED + 3
    ),
], name="training_augmentation")

def normalize_only(images, labels):
    return tf.cast(images, tf.float32) / 255.0, labels

def normalize_and_augment(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    return augmenter(images, training=True), labels

fit_dataset = fit_dataset.map(normalize_and_augment, num_parallel_calls=tf.data.AUTOTUNE)
fit_dataset = fit_dataset.prefetch(tf.data.AUTOTUNE)
validation_dataset = validation_dataset.map(normalize_only, num_parallel_calls=tf.data.AUTOTUNE)
validation_dataset = validation_dataset.prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.map(normalize_only, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

sample_indices = np.linspace(0, fit_image_tensor.shape[0] - 1, 6, dtype=int)
originals = tf.gather(fit_image_tensor, sample_indices)[..., tf.newaxis]
originals_float = tf.cast(originals, tf.float32) / 255.0
augmented = augmenter(originals_float, training=True)

figure, axes = plt.subplots(2, 6, figsize=(15, 5))
for column in range(6):
    axes[0, column].imshow(originals[column, ..., 0], cmap="gray", vmin=0, vmax=255)
    axes[1, column].imshow(augmented[column, ..., 0], cmap="gray", vmin=0, vmax=1)
    axes[0, column].axis("off")
    axes[1, column].axis("off")
    axes[0, column].set_title("Original")
    axes[1, column].set_title("Augmented")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "augmentation_preview.png", dpi=150, bbox_inches="tight")
plt.show()

images, labels = next(iter(fit_dataset))
print("Training batch:", images.shape, labels.shape, images.dtype)
print("Pixel range:", float(tf.reduce_min(images)), float(tf.reduce_max(images)))

In [ ]:
def add_classification_head(inputs, x):
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(
        num_classes, activation="softmax", dtype="float32", name="character_output"
    )(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def convolution_block(x, filters, repetitions=1):
    for _ in range(repetitions):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
    return tf.keras.layers.MaxPooling2D()(x)

def build_baseline():
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1))
    x = convolution_block(inputs, 32)
    x = convolution_block(x, 64)
    x = convolution_block(x, 128)
    return add_classification_head(inputs, x)

def build_deeper():
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1))
    x = convolution_block(inputs, 32, 2)
    x = convolution_block(x, 64, 2)
    x = convolution_block(x, 128, 2)
    x = convolution_block(x, 256, 2)
    return add_classification_head(inputs, x)

def residual_block(x, filters, stride=1):
    shortcut = x
    if stride != 1 or x.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)
    x = tf.keras.layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Add()([shortcut, x])
    return tf.keras.layers.ReLU()(x)

def build_residual():
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1))
    x = tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = residual_block(x, 32)
    x = residual_block(x, 64, 2)
    x = residual_block(x, 128, 2)
    x = residual_block(x, 256, 2)
    return add_classification_head(inputs, x)

model_builders = {
    "baseline_cnn": build_baseline,
    "deeper_cnn": build_deeper,
    "residual_cnn": build_residual,
}

for name, builder in model_builders.items():
    model = builder()
    print(name, "parameters:", f"{model.count_params():,}")
    del model

tf.keras.backend.clear_session()

In [ ]:
results_path = OUTPUT_DIR / "model_results.json"
all_results = {}
if results_path.exists():
    with results_path.open(encoding="utf-8") as file:
        all_results = json.load(file)

histories = {}

for name, builder in model_builders.items():
    print("\n" + "=" * 70)
    print("MODEL:", name)
    final_path = MODEL_DIR / f"{name}_final.keras"
    best_path = MODEL_DIR / f"{name}_best.keras"
    last_path = MODEL_DIR / f"{name}_last.keras"
    log_path = OUTPUT_DIR / f"{name}_training.csv"
    history_path = OUTPUT_DIR / f"{name}_history.json"

    if final_path.exists():
        print("Loading completed model:", final_path)
        model = tf.keras.models.load_model(final_path)
        if history_path.exists():
            with history_path.open(encoding="utf-8") as file:
                histories[name] = json.load(file)
    else:
        tf.keras.backend.clear_session()
        initial_epoch = 0
        best_val_accuracy = None
        if last_path.exists():
            print("Resuming checkpoint:", last_path)
            model = tf.keras.models.load_model(last_path)
            if log_path.exists():
                with log_path.open(newline="", encoding="utf-8") as file:
                    rows = list(csv.DictReader(file))
                    initial_epoch = len(rows)
                    if rows:
                        best_val_accuracy = max(
                            float(row["val_accuracy"]) for row in rows
                        )
        else:
            model = builder()

        model.summary()
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                best_path, monitor="val_accuracy", mode="max", save_best_only=True,
                initial_value_threshold=best_val_accuracy
            ),
            tf.keras.callbacks.ModelCheckpoint(last_path, save_best_only=False),
            tf.keras.callbacks.CSVLogger(log_path, append=True),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", mode="max", patience=2,
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=1, min_lr=1e-5
            ),
        ]

        history_data = {}
        if history_path.exists():
            with history_path.open(encoding="utf-8") as file:
                history_data = json.load(file)

        if initial_epoch < EPOCHS:
            history = model.fit(
                fit_dataset,
                validation_data=validation_dataset,
                epochs=EPOCHS,
                initial_epoch=initial_epoch,
                steps_per_epoch=STEPS_PER_EPOCH,
                validation_steps=VALIDATION_STEPS,
                callbacks=callbacks,
                verbose=1,
            )
            for key, values in history.history.items():
                history_data.setdefault(key, []).extend(float(value) for value in values)

        histories[name] = history_data
        with history_path.open("w", encoding="utf-8") as file:
            json.dump(history_data, file, indent=2)

        if best_path.exists():
            model = tf.keras.models.load_model(best_path)
        model.save(final_path)

    print("Evaluating on untouched testing images...")
    test_loss, test_accuracy = model.evaluate(
        test_dataset, steps=EVALUATION_STEPS, verbose=1
    )
    all_results[name] = {
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "model_path": str(final_path),
    }
    with results_path.open("w", encoding="utf-8") as file:
        json.dump(all_results, file, indent=2)

    del model
    gc.collect()

print("All model evaluations are complete.")

In [ ]:
ranking = sorted(
    all_results.items(), key=lambda item: item[1]["test_accuracy"], reverse=True
)

print("\nTest accuracy ranking:")
for rank, (name, result) in enumerate(ranking, 1):
    print(f"{rank}. {name}: {result['test_accuracy']:.4%}")

with (OUTPUT_DIR / "model_results.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["rank", "model", "test_accuracy", "test_loss"])
    for rank, (name, result) in enumerate(ranking, 1):
        writer.writerow([rank, name, result["test_accuracy"], result["test_loss"]])

for name, history in histories.items():
    if not history:
        continue
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.get("loss", []), label="train")
    axes[0].plot(history.get("val_loss", []), label="validation")
    axes[0].set_title(f"{name} loss")
    axes[0].legend()
    axes[1].plot(history.get("accuracy", []), label="train")
    axes[1].plot(history.get("val_accuracy", []), label="validation")
    axes[1].set_title(f"{name} accuracy")
    axes[1].legend()
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / f"{name}_history.png", dpi=150, bbox_inches="tight")
    plt.show()

best_model_name, best_result = ranking[0]
print("Best model:", best_model_name)
print("Best testing accuracy:", f"{best_result['test_accuracy']:.4%}")
print("Results saved to:", OUTPUT_DIR)

In [ ]:
best_model = tf.keras.models.load_model(best_result["model_path"])
images, labels = next(iter(test_dataset))
predictions = best_model.predict(images[:8], verbose=0)
predicted_labels = predictions.argmax(axis=1)
confidences = predictions.max(axis=1)

figure, axes = plt.subplots(2, 4, figsize=(12, 6))
for index, axis in enumerate(axes.flat):
    axis.imshow(images[index, ..., 0], cmap="gray", vmin=0, vmax=1)
    axis.axis("off")
    axis.set_title(
        f"Pred: {selected_class_ids[predicted_labels[index]]}\n"
        f"True: {selected_class_ids[int(labels[index])]}\n"
        f"Confidence: {confidences[index]:.1%}"
    )
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

print("Notebook finished. Use model_results.csv and the saved plots in your GitHub README.")